<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/Whisper_Notebooks/blob/main/Cover_Video_Subtitle_Burner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Cover Video Subtitle Burner

Combine a **still cover image**, an **audio file**, and an **SRT subtitle file** into a single `.mp4` — with subtitles burned into the frame.

---

## How To Use

### Step 1 — (Optional) Access audio from Google Drive
If your audio is in Google Drive, run the **Google Drive mount cell** first, then copy your files to `/content/input/`.

### Step 2 — Upload your three files to `/content/input/`
Open the **Files panel** (📁 left sidebar) → navigate to `/content/input/` → drag and drop:

| File | Accepted formats |
|---|---|
| Cover image | `.png` `.jpg` `.jpeg` `.webp` |
| Audio | `.mp3` `.wav` `.m4a` `.aac` `.flac` `.ogg` `.opus` `.wma` |
| Subtitle | `.srt` |

> Upload **exactly one** of each. If multiple files of the same type are present the cell stops with a clear list.

### Step 3 — Configure subtitle style (optional)
Edit the form sliders in the main code cell: font size, color, position, margins, quality.

### Step 4 — Run the main cell
Click ▶. The cell installs dependencies, preprocesses files, and renders the video.

---

## Output

- **File**: `/content/<srt-filename>.mp4` (named from your SRT file, special chars stripped)
- **Format**: H.264 video · AAC stereo audio · burned subtitles · MP4
- **Duration**: End time of the last SRT cue (extended to full audio length if audio is longer)
- **Subtitle sync**: M4A/AAC encoder delay is normalised automatically — subtitles align with speech

> ⚠️ Download your MP4 before closing the session — `/content/` is erased on disconnect.

---

## Troubleshooting

| Message | Fix |
|---|---|
| Missing file(s) | Upload all three files to `/content/input/` |
| Multiple SRT/audio files found | Remove extras — keep exactly one of each |
| `ffmpeg failed` — subtitle filter | Check `/tmp/sanitized_subtitles.srt`; ensure ffmpeg has libass |
| `ffmpeg failed` — no such file | Verify the audio path is correct |
| Session expired before download | Re-run the cell — output is regenerated |

In [ ]:
# @title ## (Optional) Mount Google Drive
# @markdown Run this cell first if your audio file is stored in Google Drive.
# @markdown After mounting, use the copy commands below to move files to `/content/input/`.

mount_drive = False  # @param {type:"boolean"}

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive/")
    print()
    print("Copy your files to the input directory, e.g.:")
    print()
    print("  import shutil, os")
    print("  os.makedirs('/content/input', exist_ok=True)")
    print("  shutil.copy('/content/drive/MyDrive/audio.m4a', '/content/input/')")
    print("  shutil.copy('/content/drive/MyDrive/cover.png', '/content/input/')")
    print("  shutil.copy('/content/drive/MyDrive/subtitles.srt', '/content/input/')")
    print()
    print("Then run the main cell below.")


In [ ]:
# @title ## Cover Video Subtitle Burner
# @markdown ### Upload cover image, audio, and .srt to `/content/input/`, then run this cell.
# @markdown Outputs an MP4 with the still image, audio, and burned-in subtitles.

# ── Phase 0: Constants & Minimal Imports ─────────────────────────────────────
# Only os and sys needed here — directory check runs before any installs.

import os
import sys

INPUT_DIR = "/content/input"
TMP_SRT   = "/tmp/sanitized_subtitles.srt"
TMP_COVER = "/tmp/cover_clean.png"

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
AUDIO_EXTS = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg', '.opus', '.wma'}
SRT_EXTS   = {'.srt'}

# ── Phase 1: Directory Check + File Detection (fail fast) ─────────────────────
# Runs instantly with no external dependencies.
# Exits immediately if files are missing or ambiguous (multiple of same type).

os.makedirs(INPUT_DIR, exist_ok=True)

_all_files = [
    f for f in os.listdir(INPUT_DIR)
    if os.path.isfile(os.path.join(INPUT_DIR, f))
]

img_files   = sorted([f for f in _all_files if os.path.splitext(f)[1].lower() in IMAGE_EXTS])
audio_files = sorted([f for f in _all_files if os.path.splitext(f)[1].lower() in AUDIO_EXTS])
srt_files   = sorted([f for f in _all_files if os.path.splitext(f)[1].lower() in SRT_EXTS])

_missing = []
if not img_files:   _missing.append("cover image  (.png / .jpg / .jpeg / .webp)")
if not audio_files: _missing.append("audio file   (.mp3 / .wav / .m4a / .aac / .flac etc.)")
if not srt_files:   _missing.append("subtitle file  (.srt)")

if _missing:
    sys.exit(
        f"\nMissing file(s) in '{INPUT_DIR}':\n" +
        "".join(f"  - {m}\n" for m in _missing) +
        "\nUpload all three files and run this cell again.\n"
    )

for _label, _found in [
    ("cover image",   img_files),
    ("audio file",    audio_files),
    ("subtitle file", srt_files),
]:
    if len(_found) > 1:
        sys.exit(
            f"[ERROR] Multiple {_label}s found — keep exactly ONE:\n" +
            "".join(f"  {f}\n" for f in _found) +
            "Remove the extras and re-run.\n"
        )

cover_path = os.path.join(INPUT_DIR, img_files[0])
audio_path = os.path.join(INPUT_DIR, audio_files[0])
srt_path   = os.path.join(INPUT_DIR, srt_files[0])

print(f"Cover : {img_files[0]}")
print(f"Audio : {audio_files[0]}")
print(f"SRT   : {srt_files[0]}")
print("Files confirmed. Loading dependencies...")

# ── Phase 2: Full Stdlib Imports ──────────────────────────────────────────────
import re
import subprocess

# ── Phase 3: Library Install ──────────────────────────────────────────────────
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--root-user-action=ignore", "Pillow", "charset-normalizer"],
    capture_output=True, text=True
)
if _pip.returncode != 0:
    print(f"[ERROR] pip install failed:\n{_pip.stderr}")
    sys.exit(1)

from PIL import Image
from charset_normalizer import from_path as detect_encoding
print("Libraries confirmed.")

# ── Phase 4: ffmpeg + libass Pre-flight ───────────────────────────────────────
if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode != 0:
    print("ffmpeg not found — installing...")
    _apt = subprocess.run(
        ["apt", "install", "-y", "-qq", "ffmpeg", "libass9", "libass-dev"],
        capture_output=True, text=True
    )
    if _apt.returncode != 0:
        print(f"[ERROR] apt install failed:\n{_apt.stderr}")
        sys.exit(1)

_filters_out = subprocess.run(["ffmpeg", "-filters"], capture_output=True, text=True)
if "subtitles" not in (_filters_out.stdout + _filters_out.stderr):
    print("[ERROR] libass/subtitles filter unavailable in this ffmpeg build.")
    print("Run in a new cell:  !apt install -y ffmpeg libass9 libass-dev")
    sys.exit(1)

_ver_line = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.split("\n")[0]
print(f"ffmpeg : {_ver_line}")
print("libass subtitles filter : confirmed.")

# ── Phase 5: Font Installation + Name Verification ───────────────────────────
# DejaVu Sans     — Latin/Greek/Cyrillic, guaranteed on Ubuntu
# Noto Sans CJK JP — Japanese/Chinese/Korean (auto-selected when SRT contains CJK)
subprocess.run(
    ["apt", "install", "-y", "-qq", "fonts-dejavu-core", "fonts-noto-cjk"],
    capture_output=True
)
subprocess.run(["fc-cache", "-fv"], capture_output=True)

_fc_list   = subprocess.run(["fc-list"], capture_output=True, text=True).stdout
FONT_LATIN = "DejaVu Sans"
FONT_CJK   = "Noto Sans CJK JP"

if FONT_LATIN not in _fc_list:
    print(f"[WARN] '{FONT_LATIN}' not in fontconfig — subtitles may use fallback font.")
else:
    print(f"Font confirmed : {FONT_LATIN}")

if FONT_CJK not in _fc_list:
    print(f"[WARN] '{FONT_CJK}' not in fontconfig — CJK subtitles may not render.")
else:
    print(f"Font confirmed : {FONT_CJK}")

# ── Phase 6: File Validation ──────────────────────────────────────────────────

# 6a — Cover image integrity check
try:
    _img_test = Image.open(cover_path)
    _img_test.verify()
except Exception as _e:
    sys.exit(f"[ERROR] Cover image corrupt or unreadable: {_e}")

# 6b — Audio: ffprobe for duration (also validates the file is readable)
_audio_probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        audio_path,
    ],
    capture_output=True, text=True,
)
if _audio_probe.returncode != 0:
    sys.exit(
        f"[ERROR] Audio file corrupt or unsupported:\n{_audio_probe.stderr.strip()}"
    )
try:
    audio_duration_secs = float(_audio_probe.stdout.strip())
    print(f"Audio duration : {audio_duration_secs:.3f}s")
except ValueError:
    audio_duration_secs = None
    print("[WARN] Could not read audio duration — will use SRT duration.")

# 6c — SRT non-empty check
if os.path.getsize(srt_path) == 0:
    sys.exit("[ERROR] SRT file is empty.")

print("File validation passed.")

# ── Phase 7: SRT Preprocessing ────────────────────────────────────────────────
# Normalises encoding, line endings, timing separators, and strips problematic
# markup. Writes a clean UTF-8 SRT to a safe ASCII temp path, eliminating every
# class of ffmpeg filter path-escaping issue.

# 7a — Detect encoding and decode
_detected = detect_encoding(srt_path).best()
if _detected is None:
    _raw_bytes = open(srt_path, "rb").read()
    for _enc in ("utf-8-sig", "utf-8", "latin-1", "cp1252", "shift_jis"):
        try:
            _content = _raw_bytes.decode(_enc)
            break
        except (UnicodeDecodeError, LookupError):
            continue
    else:
        sys.exit("[ERROR] Cannot detect SRT file encoding.")
else:
    _content = str(_detected)   # charset_normalizer strips BOM automatically

# 7b — Normalise line endings (CRLF and bare CR → LF)
_content = _content.replace("\r\n", "\n").replace("\r", "\n")

# 7c — Normalise timing separator: some exporters use HH:MM:SS.mmm not HH:MM:SS,mmm
_TIMING_PERIOD = re.compile(
    r"^(\d{2}:\d{2}:\d{2})\.(\d{3}\s*-->\s*\d{2}:\d{2}:\d{2})\.(\d{3})",
    re.MULTILINE,
)
_content = _TIMING_PERIOD.sub(r"\1,\2,\3", _content)

# 7d — Strip ASS override tags and problematic HTML from subtitle text lines.
#      Timing lines and sequence-number lines are left untouched.
#      <i>, <b>, <u> are preserved (libass handles them natively).
_IS_TIMING = re.compile(
    r"^\d{2}:\d{2}:\d{2},\d{3}\s*-->\s*\d{2}:\d{2}:\d{2},\d{3}"
)
_IS_SEQ = re.compile(r"^\d+\s*$")


def _clean_line(line: str) -> str:
    line = re.sub(r"\{[^}]*\}", "", line)                           # {\\an8} ASS tags
    line = re.sub(r"<font[^>]*>", "", line, flags=re.IGNORECASE)    # <font ...>
    line = re.sub(r"</font>",     "", line, flags=re.IGNORECASE)    # </font>
    # Remove all HTML except <i>, </i>, <b>, </b>, <u>, </u>
    line = re.sub(r"<(?!/?(?:i|b|u)>)[^>]+>", "", line)
    return line


_cleaned_lines = []
for _ln in _content.split("\n"):
    if _IS_TIMING.match(_ln.strip()) or _IS_SEQ.match(_ln.strip()):
        _cleaned_lines.append(_ln)
    else:
        _cleaned_lines.append(_clean_line(_ln))
_content = "\n".join(_cleaned_lines)

# 7e — Collapse excess blank lines (3+ consecutive blank lines → 1)
_content = re.sub(r"\n{3,}", "\n\n", _content).strip() + "\n"

# 7f — Parse last cue end time to determine SRT total duration
_END_TIME_RE = re.compile(
    r"\d{2}:\d{2}:\d{2},\d{3}\s*-->\s*(\d{2}:\d{2}:\d{2},\d{3})"
)


def _srt_to_secs(t: str) -> float:
    t = t.strip().replace(",", ".")
    h, m, rest = t.split(":")
    s, ms = rest.split(".")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0


_end_times = [_srt_to_secs(m) for m in _END_TIME_RE.findall(_content)]
if not _end_times:
    sys.exit("[ERROR] No valid subtitle timing lines found in SRT file.")

srt_duration_secs = max(_end_times)
_srt_mm = int(srt_duration_secs // 60)
_srt_ss = int(srt_duration_secs % 60)
print(f"SRT duration : {srt_duration_secs:.3f}s  ({_srt_mm:02d}:{_srt_ss:02d})")

# 7g — Write sanitized SRT to safe ASCII temp path
with open(TMP_SRT, "w", encoding="utf-8") as _f:
    _f.write(_content)
print(f"SRT sanitized → {TMP_SRT}")

# ── Phase 8: Image Preprocessing ─────────────────────────────────────────────
# Converts to RGB (libx264 cannot accept RGBA/palette modes).
# Composites transparent images onto a black background.
# Enforces even pixel dimensions (libx264 hard requirement).

img = Image.open(cover_path)

if img.mode in ("RGBA", "LA"):
    _bg = Image.new("RGB", img.size, (0, 0, 0))
    _bg.paste(img, mask=img.split()[-1])
    img = _bg
elif img.mode in ("PA", "P"):
    _rgba = img.convert("RGBA")
    _bg = Image.new("RGB", img.size, (0, 0, 0))
    _bg.paste(_rgba, mask=_rgba.split()[-1])
    img = _bg
elif img.mode != "RGB":
    img = img.convert("RGB")

w, h = img.size
w = w if w % 2 == 0 else w - 1
h = h if h % 2 == 0 else h - 1
if (w, h) != img.size:
    img = img.crop((0, 0, w, h))

img_width, img_height = w, h
img.save(TMP_COVER, "PNG")
print(f"Cover preprocessed : {img_width}x{img_height} → {TMP_COVER}")

# ── Phase 9: Duration Resolution + Output Path ────────────────────────────────

video_duration = srt_duration_secs

if audio_duration_secs is not None:
    if audio_duration_secs > srt_duration_secs:
        video_duration = audio_duration_secs
        print(
            f"[INFO] Audio ({audio_duration_secs:.1f}s) longer than SRT "
            f"({srt_duration_secs:.1f}s) — extending video to audio length."
        )
    elif audio_duration_secs < srt_duration_secs - 1.0:
        print(
            f"[INFO] Audio ({audio_duration_secs:.1f}s) shorter than SRT "
            f"({srt_duration_secs:.1f}s) — silence will pad the end."
        )


def _safe_stem(path: str) -> str:
    stem = os.path.splitext(os.path.basename(path))[0]
    safe = "".join(
        c if c.isascii() and (c.isalnum() or c in "-_. ") else "_"
        for c in stem
    ).strip("_")
    return safe[:80] or "output"


OUTPUT_PATH = f"/content/{_safe_stem(srt_path)}.mp4"
print(f"Output duration : {video_duration:.3f}s")
print(f"Output path     : {OUTPUT_PATH}")

# ── Phase 10: User Style Parameters ──────────────────────────────────────────
# @markdown ---
# @markdown ## Output Settings
output_resolution = "Original"  # @param ["Original", "1920x1080", "1280x720", "854x480"]
video_crf         = 23          # @param {type:"slider", min:18, max:35, step:1}
video_quality     = "medium"    # @param ["fast", "medium", "slow"]

# @markdown ## Subtitle Style
font_size    = 28       # @param {type:"slider", min:12, max:72, step:2}
font_color   = "white"  # @param ["white", "yellow", "cyan", "black", "green"]
outline_size = 2        # @param {type:"slider", min:0, max:5, step:1}
shadow_size  = 1        # @param {type:"slider", min:0, max:3, step:1}
position     = "bottom" # @param ["bottom", "top"]
margin_v     = 25       # @param {type:"slider", min:0, max:150, step:5}
margin_lr    = 40       # @param {type:"slider", min:0, max:200, step:10}

# ASS color format: &HAABBGGRR& — AA=00 means fully opaque
_COLOR_MAP = {
    "white":  "&H00FFFFFF&",
    "yellow": "&H0000FFFF&",   # BGR: 00-FF-FF → RGB(255,255,0)
    "cyan":   "&H00FFFF00&",   # BGR: FF-FF-00 → RGB(0,255,255)
    "black":  "&H00000000&",
    "green":  "&H0000FF00&",   # BGR: 00-FF-00 → RGB(0,255,0)
}
_primary_colour = _COLOR_MAP[font_color]
_outline_colour = "&H00000000&"            # black outline always
_alignment      = 2 if position == "bottom" else 8  # 2=centre-bottom, 8=centre-top

# Output dimensions (enforce even numbers for libx264)
if output_resolution == "Original":
    _out_w, _out_h = img_width, img_height
else:
    _out_w, _out_h = map(int, output_resolution.split("x"))

_out_w = _out_w if _out_w % 2 == 0 else _out_w - 1
_out_h = _out_h if _out_h % 2 == 0 else _out_h - 1


def _has_cjk(text: str) -> bool:
    for ch in text:
        cp = ord(ch)
        if (0x3000 <= cp <= 0x9FFF
                or 0xF900 <= cp <= 0xFAFF
                or 0x20000 <= cp <= 0x2FA1F):
            return True
    return False


with open(TMP_SRT, encoding="utf-8") as _f:
    _srt_text = _f.read()

_font_name = FONT_CJK if _has_cjk(_srt_text) else FONT_LATIN
print(f"Font selected : {_font_name}")

# ── Phase 11: ffmpeg Command Build + Execute ──────────────────────────────────

# force_style: PlayResX/Y set the coordinate space so MarginV/L/R are in actual
# output pixels. WrapStyle=0 enables smart word-wrap for long subtitle lines.
_force_style = (
    f"PlayResX={_out_w},"
    f"PlayResY={_out_h},"
    f"FontName={_font_name},"
    f"FontSize={font_size},"
    f"PrimaryColour={_primary_colour},"
    f"OutlineColour={_outline_colour},"
    f"BorderStyle=1,"
    f"Outline={outline_size},"
    f"Shadow={shadow_size},"
    f"WrapStyle=0,"
    f"Alignment={_alignment},"
    f"MarginV={margin_v},"
    f"MarginL={margin_lr},"
    f"MarginR={margin_lr}"
)

# TMP_SRT = /tmp/sanitized_subtitles.srt — no colons, spaces, or special chars.
# charenc=UTF-8 overrides any libass internal encoding detection attempt.
_subtitle_filter = (
    f"subtitles='{TMP_SRT}'"
    f":charenc=UTF-8"
    f":force_style='{_force_style}'"
)

# Scale filter — two passes guarantee even dimensions through all code paths:
# Pass 1: fit image within target dimensions (letterbox/pillarbox if needed)
# Pass 2: trunc(x/2)*2 forces even width and height (libx264 hard requirement)
if output_resolution == "Original":
    _scale_filter = "scale=trunc(iw/2)*2:trunc(ih/2)*2"
else:
    _scale_filter = (
        f"scale={_out_w}:{_out_h}:force_original_aspect_ratio=decrease,"
        f"scale=trunc(iw/2)*2:trunc(ih/2)*2,"
        f"pad={_out_w}:{_out_h}:(ow-iw)/2:(oh-ih)/2:black"
    )

_vf = f"{_scale_filter},format=yuv420p,{_subtitle_filter}"

cmd = [
    "ffmpeg", "-y",
    # Image input at 1 fps — H.264 encodes repeated identical frames as
    # near-zero-size P-frames, making long static-image videos encode fast.
    "-framerate", "1", "-loop", "1", "-i", TMP_COVER,
    # Audio input — path may contain any Unicode/special chars; safe in list mode.
    "-i", audio_path,
    # Video filter chain: scale → even-dims → yuv420p → burned subtitles
    "-vf", _vf,
    # Audio filters:
    #   asetpts=PTS-STARTPTS  normalises M4A/AAC ELST edit-list offset to t=0,
    #                         keeping subtitle timings aligned with speech.
    #   apad                  pads with silence if audio ends before -t duration.
    "-af", "asetpts=PTS-STARTPTS,apad",
    # Output at standard 25 fps (interpolated from 1 fps input)
    "-r", "25",
    "-c:v", "libx264", "-crf", str(video_crf), "-preset", video_quality,
    # Explicit stereo AAC — avoids channel-layout warnings on some M4A inputs
    "-c:a", "aac", "-b:a", "192k", "-ac", "2",
    # Required for Apple / QuickTime / browser compatibility
    "-pix_fmt", "yuv420p",
    # Hard output duration: prevents infinite loop from -loop 1
    "-t", str(video_duration),
    # Move moov atom to front of file (enables progressive streaming)
    "-movflags", "+faststart",
    OUTPUT_PATH,
]

_mm = int(video_duration // 60)
_ss = int(video_duration % 60)
print(
    f"\nRendering {_mm:02d}:{_ss:02d} at {_out_w}x{_out_h} | "
    f"CRF {video_crf} | preset={video_quality} | font={_font_name}\n"
)

_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)
_last_lines = []
for _line in _proc.stdout:
    _line = _line.rstrip()
    if _line.startswith("frame=") or _line.startswith("size="):
        print(f"\r{_line}", end="", flush=True)
    elif _line:
        _last_lines.append(_line)
        if len(_last_lines) > 30:
            _last_lines.pop(0)
_proc.wait()
print()

if _proc.returncode != 0:
    _tail = "\n".join(_last_lines)
    print(f"[ERROR] ffmpeg exited with code {_proc.returncode}")
    print(f"Last output:\n{_tail}")
    if "Invalid option" in _tail or "Unknown encoder" in _tail:
        print("\nHint: encoder unavailable — run:  !apt install -y ffmpeg")
    elif "No such file" in _tail:
        print("\nHint: input file not found — check /content/input/ contents.")
    elif "subtitles" in _tail.lower() or "ass" in _tail.lower():
        print("\nHint: subtitle filter error — inspect /tmp/sanitized_subtitles.srt")
        print("      Likely cause: libass not compiled into this ffmpeg build.")
    elif "Invalid data" in _tail or "moov atom" in _tail:
        print("\nHint: audio file may be corrupt or in an unsupported format.")
    elif "not divisible" in _tail:
        print("\nHint: dimension parity error — please report this bug.")
    sys.exit(1)

# ── Phase 12: Output Verification + Download Prompt ──────────────────────────

if not os.path.exists(OUTPUT_PATH) or os.path.getsize(OUTPUT_PATH) == 0:
    sys.exit("[ERROR] Output file was not created or is empty.")

_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

_verify = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "stream=codec_type,codec_name,width,height,duration",
        "-of", "default=noprint_wrappers=1",
        OUTPUT_PATH,
    ],
    capture_output=True, text=True,
)

print(f"\n✓  {OUTPUT_PATH}  ({_size_mb:.2f} MB)")
print(_verify.stdout.strip())
print()
print("Download: right-click the file in the Files panel (📁 left sidebar) → Download")
print("⚠️  Download before closing the session — /content/ is erased on disconnect.")
